In [ ]:
import dash
from dash import dcc, html, Input, Output, State
import plotly.express as px
import pandas as pd
import json
from urllib.request import urlopen
import base64
import sqlite3
import os

In [ ]:
# Load geographical/map data
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

In [ ]:
# Connect to database to fetch metadata
db_path = r'C:\Users\bthem\Documents\AI_Masters\GRA\pr_dashboard.db'

def get_db_connection():
    return sqlite3.connect(db_path)

conn = get_db_connection()
# Get all tables except municipalities, sqlite_sequence, and the master score
tables_query = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT IN ('municipalities', 'sqlite_sequence', 'Overall_MHVI_M_Score')"
tables = pd.read_sql_query(tables_query, conn)['name'].tolist()

db_metadata = {}
for tbl in tables:
    try:
        inds = pd.read_sql_query(f"SELECT DISTINCT indicator_name FROM {tbl}", conn)['indicator_name'].tolist()
        if inds:
            db_metadata[tbl] = inds
    except Exception as e:
        pass
conn.close()

available_subcats = list(db_metadata.keys())

In [ ]:
# Define global overarching categories
global_categories = ['Overall MHVI-M Score'] + available_subcats

In [ ]:
# Helper to encode local images
def encode_image(image_path):
    try:
        with open(image_path, 'rb') as f:
            encoded = base64.b64encode(f.read()).decode('ascii')
        return f'data:image/png;base64,{encoded}'
    except FileNotFoundError:
        return ""

logo1_path = r'C:\Users\bthem\Documents\AI_Masters\GRA\MB_Horz_3Clr.png'
logo2_path = r'C:\Users\bthem\Documents\AI_Masters\GRA\Grupo_nexos.png'
encoded_logo1 = encode_image(logo1_path)
encoded_logo2 = encode_image(logo2_path)

In [ ]:
# Initialize Dash App
app = dash.Dash(__name__)

menu_btn_style = {
    'width': '100%', 'padding': '10px', 'marginBottom': '10px', 
    'backgroundColor': '#ffffff', 'border': '1px solid #ccc', 
    'borderRadius': '4px', 'cursor': 'pointer', 'textAlign': 'left',
    'fontWeight': 'bold', 'color': '#333'
}

dropdown_style = {'width': '100%', 'marginBottom': '15px'}

In [ ]:
# Define Layout
app.layout = html.Div([
    
    # Header Container
    html.Div([
        html.Div([
            html.Button("☰", id="left-menu-btn", style={
                'fontSize': '24px', 'backgroundColor': 'transparent', 
                'border': 'none', 'cursor': 'pointer', 'padding': '10px', 'marginRight': '15px'
            }),
            html.Div([
                html.H1("Puerto Rico Mental Health Vulnerability Index for Minors", style={'margin': '0', 'fontFamily': 'Arial'}),
            ])
        ], style={'display': 'flex', 'alignItems': 'center'}),
        
        html.Div([
            html.Img(src=encoded_logo1, style={'height': '50px', 'objectFit': 'contain'}),
            html.Span(" | ", style={'fontSize': '30px', 'color': '#ccc', 'margin': '0 15px'}),
            html.Img(src=encoded_logo2, style={'height': '50px', 'objectFit': 'contain'})
        ], style={'display': 'flex', 'alignItems': 'center'})
        
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'space-between', 'padding': '10px 20px', 'borderBottom': '1px solid #ccc'}),
    
    # General Controls (Top Bar)
    html.Div([
        html.Div([
            html.Label("Category:", style={'fontWeight': 'bold', 'marginRight': '10px'}),
            dcc.Dropdown(
                id='global-category-dropdown',
                options=[{'label': cat.replace('_', ' '), 'value': cat} for cat in global_categories],
                value='Overall MHVI-M Score',
                clearable=False,
                style={'width': '350px'}
            )
        ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'}),
    ], style={'display': 'flex', 'padding': '10px 20px', 'backgroundColor': '#f0f2f5', 'borderBottom': '1px solid #ccc', 'fontFamily': 'Arial'}),

    # Main Content Container (Flexbox)
    html.Div([
        
        # Left column: Menu
        html.Div(
            id='left-menu-container',
            style={'display': 'none'}, 
            children=[
                html.H3("Navigation", style={'borderBottom': '2px solid #ccc', 'paddingBottom': '10px'}),
                html.Button("Custom Reports", style=menu_btn_style),
                html.Button("Publication", style=menu_btn_style),
                html.Button("Data Sources", style=menu_btn_style),
                html.Button("Source Code", style=menu_btn_style)
            ]
        ),
        
        # Center column: Interactive Map
        html.Div([
            dcc.Graph(
                id='pr-map', 
                style={'height': '75vh'}
            )
        ], style={'flex': '7', 'padding': '10px', 'minWidth': '0'}), 
        
        # Right column: Data Panel
        html.Div(
            id='side-panel-container', 
            style={'display': 'none'}, 
            children=[
                html.Button("✖ Close", id="close-panel-btn", style={
                    'float': 'right', 'backgroundColor': '#ff4d4d', 'color': 'white', 
                    'border': 'none', 'padding': '5px 10px', 'borderRadius': '4px', 'cursor': 'pointer'
                }),
                html.H2(id='county-title', style={'borderBottom': '2px solid #ccc', 'paddingBottom': '10px', 'marginTop': '0'}),
                
                # 2x2 Factor Matrix
                html.Div([
                    # Left Column: Y-Axis
                    html.Div([
                        html.Label("Y-Axis Category", style={'fontWeight': 'bold', 'fontSize': '14px'}),
                        dcc.Dropdown(
                            id='y-cat-dropdown',
                            options=[{'label': c.replace('_', ' '), 'value': c} for c in global_categories],
                            value='Overall MHVI-M Score', clearable=False, style=dropdown_style
                        ),
                        html.Label("Y-Axis Indicator", style={'fontWeight': 'bold', 'fontSize': '14px', 'color': '#666'}),
                        dcc.Dropdown(
                            id='y-ind-dropdown',
                            options=[{'label': 'N/A', 'value': 'N/A'}],
                            value='N/A', disabled=True, clearable=False, style=dropdown_style
                        ),
                    ], style={'flex': '1', 'marginRight': '15px'}),
                    
                    # Right Column: Additional Indicator (Color)
                    html.Div([
                        html.Label("Additional Indicator (Color)", style={'fontWeight': 'bold', 'fontSize': '14px'}),
                        dcc.Dropdown(
                            id='color-cat-dropdown',
                            options=[{'label': c.replace('_', ' '), 'value': c} for c in ['N/A'] + global_categories],
                            value='N/A', clearable=False, style=dropdown_style
                        ),
                        html.Label("Indicator Selection", style={'fontWeight': 'bold', 'fontSize': '14px', 'color': '#666'}),
                        dcc.Dropdown(
                            id='color-ind-dropdown',
                            options=[{'label': 'N/A', 'value': 'N/A'}],
                            value='N/A', disabled=True, clearable=False, style=dropdown_style
                        ),
                    ], style={'flex': '1'})
                ], style={'display': 'flex', 'flexDirection': 'row', 'marginBottom': '20px'}),
                
                html.Hr(),
                
                html.Div([
                    dcc.Graph(id='factor-trend-graph', style={'height': '40vh'})
                ], style={'marginBottom': '20px'})
                
            ]
        )
    ], style={'display': 'flex', 'flexDirection': 'row'}) 
], style={'fontFamily': 'Arial'})

db_metadata_json = json.dumps(db_metadata)

In [ ]:
# Callbacks
@app.callback(
    Output('pr-map', 'figure'),
    Input('global-category-dropdown', 'value')
)
def update_map(category):
    conn = get_db_connection()
    if category == 'Overall MHVI-M Score':
        target_table = 'Overall_MHVI_M_Score'
        ind = 'Overall MHVI-M Score'
    else:
        target_table = category
        ind = 'Subcategory Index Score'
    
    try:
        df = pd.read_sql(f"""
            SELECT m.fips_code, m.name, t.value 
            FROM {target_table} t 
            JOIN municipalities m ON t.fips_code = m.fips_code 
            WHERE t.indicator_name = '{ind}' AND t.year = (SELECT MAX(year) FROM {target_table})
        """, conn)
    except:
        df = pd.DataFrame(columns=['fips_code', 'value', 'name'])
    finally:
        conn.close()
        
    if df.empty:
        fig = px.choropleth_map(geojson=counties, locations=[], map_style="carto-positron", zoom=7.5, center={"lat": 18.2208, "lon": -66.5901})
        fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
        return fig
        
    fig = px.choropleth_map(
        df,
        geojson=counties,
        locations='fips_code',
        color='value',
        hover_name='name',
        color_continuous_scale="Viridis_r",
        map_style="carto-positron",
        zoom=7.5,
        center={"lat": 18.2208, "lon": -66.5901},
        opacity=0.8,
        hover_data={'fips_code': False, 'value': True}
    )
    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
    return fig

In [ ]:
@app.callback(
    [Output('y-ind-dropdown', 'options'), Output('y-ind-dropdown', 'value'), Output('y-ind-dropdown', 'disabled')],
    Input('y-cat-dropdown', 'value')
)
def update_y_indicator(y_cat):
    if y_cat in ['N/A', 'Overall MHVI-M Score']:
        return [{'label': 'N/A', 'value': 'N/A'}], 'N/A', True
        
    metadata = json.loads(db_metadata_json)
    indicators = metadata.get(y_cat, [])
    options = [{'label': i.replace('_', ' '), 'value': i} for i in indicators]
    val = indicators[0] if indicators else 'N/A'
    return options, val, False

In [ ]:
@app.callback(
    [Output('color-ind-dropdown', 'options'), Output('color-ind-dropdown', 'value'), Output('color-ind-dropdown', 'disabled')],
    Input('color-cat-dropdown', 'value')
)
def update_color_indicator(color_cat):
    if color_cat in ['N/A', 'Overall MHVI-M Score']:
        return [{'label': 'N/A', 'value': 'N/A'}], 'N/A', True
        
    metadata = json.loads(db_metadata_json)
    indicators = metadata.get(color_cat, [])
    options = [{'label': i.replace('_', ' '), 'value': i} for i in indicators]
    val = indicators[0] if indicators else 'N/A'
    return options, val, False

In [ ]:
@app.callback(
    Output('left-menu-container', 'style'),
    Input('left-menu-btn', 'n_clicks'),
    State('left-menu-container', 'style') 
)
def toggle_left_menu(n_clicks, current_style):
    if n_clicks is None:
        raise dash.exceptions.PreventUpdate
    open_style = {
        'flex': '2', 'padding': '20px', 'backgroundColor': '#f0f2f5', 
        'borderRight': '1px solid #ccc', 'fontFamily': 'Arial', 
        'height': '68vh', 'display': 'flex', 'flexDirection': 'column',
        'minWidth': '0'
    }
    if current_style and current_style.get('display') == 'none':
        return open_style
    else:
        return {'display': 'none'}

In [ ]:
@app.callback(
    Output('side-panel-container', 'style'),
    [Input('pr-map', 'clickData'), 
     Input('close-panel-btn', 'n_clicks')]
)
def toggle_right_panel(map_click, close_click):
    trigger_id = dash.ctx.triggered_id
    if not trigger_id:
        return {'display': 'none'} 
    open_style = {
        'flex': '4', 'padding': '20px', 'backgroundColor': '#f8f9fa', 
        'borderLeft': '1px solid #ccc', 'fontFamily': 'Arial', 
        'height': '75vh', 'overflowY': 'auto',
        'display': 'flex', 'flexDirection': 'column',
        'minWidth': '0'
    }
    if trigger_id == 'close-panel-btn':
        return {'display': 'none'}
    elif trigger_id == 'pr-map' and map_click is not None:
        return open_style
    return {'display': 'none'}

In [ ]:
def query_axis_data(fips, cat, ind):
    conn = get_db_connection()
    if cat == 'Year':
        try:
            years_df = pd.read_sql("SELECT DISTINCT year FROM Economic_Employment ORDER BY year", conn) 
            years_df['val'] = years_df['year']
            return years_df
        except:
            return pd.DataFrame({'year': [2018, 2019, 2020, 2021, 2022, 2023, 2024], 'val': [2018, 2019, 2020, 2021, 2022, 2023, 2024]})
        finally:
            conn.close()
    elif cat == 'Overall MHVI-M Score':
        try:
            df = pd.read_sql(f"SELECT year, value as val FROM Overall_MHVI_M_Score WHERE fips_code = '{fips}'", conn)
        except:
            df = pd.DataFrame(columns=['year', 'val'])
        finally:
            conn.close()
        return df
    else:
        try:
            df = pd.read_sql(f"SELECT year, value as val FROM {cat} WHERE fips_code = '{fips}' AND indicator_name = '{ind}'", conn)
        except:
            df = pd.DataFrame(columns=['year', 'val'])
        finally:
            conn.close()
        return df

In [ ]:
@app.callback(
    [Output('county-title', 'children'), Output('factor-trend-graph', 'figure')],
    [Input('pr-map', 'clickData'), Input('y-cat-dropdown', 'value'), Input('y-ind-dropdown', 'value'),
     Input('color-cat-dropdown', 'value'), Input('color-ind-dropdown', 'value')]
)
def update_dynamic_plot(clickData, y_cat, y_ind, color_cat, color_ind):
    if clickData is None:
        raise dash.exceptions.PreventUpdate
        
    clicked_fips = clickData['points'][0]['location']
    
    conn = get_db_connection()
    try:
        muni_name = pd.read_sql_query(f"SELECT name FROM municipalities WHERE fips_code = '{clicked_fips}'", conn).iloc[0]['name']
    except:
        muni_name = "Unknown Municipality"
    finally:
        conn.close()
        
    # Base Y-Axis query    
    df_y = query_axis_data(clicked_fips, y_cat, y_ind)
    
    if df_y.empty:
        fig = px.scatter(title="No Y-Axis Data Found")
        return muni_name, fig
        
    df_y = df_y.sort_values(by='year')
    label_y = y_ind.replace('_', ' ') if y_cat not in ['Overall MHVI-M Score'] else y_cat
    
    # Color Query Logic
    if color_cat != 'N/A':
        df_color = query_axis_data(clicked_fips, color_cat, color_ind)
        df_merged = pd.merge(df_y, df_color, on='year', how='left', suffixes=('_y', '_color'))
        
        label_color = color_ind.replace('_', ' ') if color_cat not in ['Overall MHVI-M Score'] else color_cat
        fig = px.scatter(
            df_merged, x='year', y='val_y', color='val_color',
            color_continuous_scale='Viridis_r', labels={'val_color': label_color}
        )
        # Force the scatter into a continuous line graph with colored nodes
        fig.update_traces(mode='lines+markers', line=dict(color='gray', width=2), marker=dict(size=12))
        
        # Rotate the colorbar title to sit sideways down the height of the figure
        fig.update_layout(coloraxis_colorbar=dict(title_side="right"))
        
    else:
        # Standard line chart with no complex color grading
        fig = px.line(df_y, x='year', y='val', markers=True)
        fig.update_traces(marker=dict(size=10, color='#e76f51'), line=dict(color='#e76f51'))

    fig.update_layout(
        margin={"r":0,"t":10,"l":0,"b":0}, 
        paper_bgcolor='#f8f9fa', plot_bgcolor='#f8f9fa', 
        xaxis=dict(title="Year", type='category'), 
        yaxis=dict(title=label_y)
    )
    return muni_name, fig

In [ ]:
 df_y = query_axis_data(clicked_fips, y_cat, y_ind)
    
    if df_y.empty:
        fig = px.scatter(title="No Y-Axis Data Found")
        return muni_name, fig
        
    df_y = df_y.sort_values(by='year')
    label_y = y_ind.replace('_', ' ') if y_cat not in ['Overall MHVI-M Score'] else y_cat
    
    # Color Query Logic
    if color_cat != 'N/A':
        df_color = query_axis_data(clicked_fips, color_cat, color_ind)
        df_merged = pd.merge(df_y, df_color, on='year', how='left', suffixes=('_y', '_color'))
        
        label_color = color_ind.replace('_', ' ') if color_cat not in ['Overall MHVI-M Score'] else color_cat
        fig = px.scatter(
            df_merged, x='year', y='val_y', color='val_color',
            color_continuous_scale='Viridis_r', labels={'val_color': label_color}
        )
        # Force the scatter into a continuous line graph with colored nodes
        fig.update_traces(mode='lines+markers', line=dict(color='gray', width=2), marker=dict(size=12))
        
        # Rotate the colorbar title to sit sideways down the height of the figure
        fig.update_layout(coloraxis_colorbar=dict(title_side="right"))
        
    else:
        # Standard line chart with no complex color grading
        fig = px.line(df_y, x='year', y='val', markers=True)
        fig.update_traces(marker=dict(size=10, color='#e76f51'), line=dict(color='#e76f51'))

    fig.update_layout(
        margin={"r":0,"t":10,"l":0,"b":0}, 
        paper_bgcolor='#f8f9fa', plot_bgcolor='#f8f9fa', 
        xaxis=dict(title="Year", type='category'), 
        yaxis=dict(title=label_y)
    )
    return muni_name, fig